In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

In [2]:
def applay_controlled_U(qc, n, angle):
    repetitions = 1
    for qubit in range(n):
        for repetition in range(repetitions):
            qc.cp(angle, qubit, n)
        repetitions *= 2

In [3]:
def iqft(n):
    qc = QuantumCircuit(n)
    for target in reversed(range(n)):
        for control in reversed(range(target +1, n)):
            k = control - target + 1
            angle = -2 * np.pi / (2 ** k)
            qc.cp(angle, control, target)
        qc.h(target)
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
    return qc

In [4]:
def run_qpe(n, theta):
    n_state = 1
    qc = QuantumCircuit(n + n_state, n)

    for i in range(n):
        qc.h(i)
    
    qc.x(n)
    qc.barrier()

    angle = 2 * np.pi * theta
    applay_controlled_U(qc, n, angle)
    qc.barrier()

    iqft_circuit = iqft(n)
    qc.compose(iqft_circuit, range(n), inplace=True)

    qc.measure(range(n), range(n))  

    return qc

In [5]:
n = 3
circuit = run_qpe(n, 0.375)

simulator = AerSimulator()
result = simulator.run(circuit, shots=1024).result()
counts = result.get_counts()

print(f"Measurement results: {counts}")

Measurement results: {'011': 1024}
